* The Scenario: You are given three separate datasets: "User Demographics", "Track Metadata" (artist, genre, duration), and "Listening History" (who listened to what, and when).

In [50]:
import pandas as pd
users = pd.read_csv("user_demographics.csv")
tracks = pd.read_csv("track_metadata.csv")
history_q1 = pd.read_csv("listening_history_q1.csv")
history_q2 = pd.read_csv("listening_history_q2.csv")

* Combining Datasets: You must use pd.concat() to stitch together listening history from Q1 and Q2. Then, use pd.merge() to join the 'Listening History' table with the 'Track Metadata' table so you know the genre of the song played, acting like a SQL left join.

In [57]:
combined = pd.concat([history_q1, history_q2], ignore_index=True)
combined
# print(combined.columns)
# print(tracks.columns)
merge_df = pd.merge(combined, tracks, left_on='trk_ref_id', right_on='track_id', how='inner')
merge_df

,listen_id,usr_ref_id,trk_ref_id,timestamp,track_id,artist_name,Genre,length_in_ms,internal_db_id
0,1,167,146,2025-01-23 22:19:45,146,Artist_46,Electronic,121062,119870
1,3,90,105,2025-01-06 15:32:21,105,Artist_5,Electronic,285983,138102
2,5,39,119,2025-02-12 11:12:04,119,Artist_19,Hip-Hop,134397,625830
3,6,126,149,2025-03-05 19:52:42,149,Artist_49,Pop,272617,334677
4,8,173,101,2025-01-04 5:56:14,101,Artist_1,Pop,205999,897606
...,...,...,...,...,...,...,...,...,...
1495,1494,175,126,2025-06-15 7:30:32,126,Artist_26,Rock,298352,319930
1496,1495,165,101,2025-05-22 13:18:12,101,Artist_1,Pop,205999,897606
1497,1496,49,128,2025-06-13 22:29:31,128,Artist_28,Classical,267851,617313
1498,1497,120,137,2025-04-26 2:20:18,137,Artist_37,Hip-Hop,259407,257504


* Modifying DataFrames: You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. Use .drop() to clear redundant system ID columns and .rename() poorly named columns.

In [58]:
Minutes_Played = merge_df["length_in_ms"] / 60000
Minutes_Played

0       2.017700
1       4.766383
2       2.239950
3       4.543617
4       3.433317
          ...   
1495    4.972533
1496    3.433317
1497    4.464183
1498    4.323450
1499    4.592300
Name: length_in_ms, Length: 1500, dtype: float64

In [59]:
clear = combined.drop(columns=["listen_id"], errors="ignore")
clear
combined.rename(columns={"ts": "timestamp"}, inplace=True)
combined

,listen_id,usr_ref_id,trk_ref_id,timestamp
0,1,167,146,2025-01-23 22:19:45
1,3,90,105,2025-01-06 15:32:21
2,5,39,119,2025-02-12 11:12:04
3,6,126,149,2025-03-05 19:52:42
4,8,173,101,2025-01-04 5:56:14
...,...,...,...,...
1495,1494,175,126,2025-06-15 7:30:32
1496,1495,165,101,2025-05-22 13:18:12
1497,1496,49,128,2025-06-13 22:29:31
1498,1497,120,137,2025-04-26 2:20:18


* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the timestamp.

In [60]:
combined["timestamp"] = pd.to_datetime(combined["timestamp"])
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"
combined["Time_of_Day"] = combined["timestamp"].dt.hour.apply(get_time_of_day)
print(combined)

      listen_id  usr_ref_id  trk_ref_id           timestamp Time_of_Day
0             1         167         146 2025-01-23 22:19:45       Night
1             3          90         105 2025-01-06 15:32:21   Afternoon
2             5          39         119 2025-02-12 11:12:04     Morning
3             6         126         149 2025-03-05 19:52:42     Evening
4             8         173         101 2025-01-04 05:56:14     Morning
...         ...         ...         ...                 ...         ...
1495       1494         175         126 2025-06-15 07:30:32     Morning
1496       1495         165         101 2025-05-22 13:18:12   Afternoon
1497       1496          49         128 2025-06-13 22:29:31       Night
1498       1497         120         137 2025-04-26 02:20:18       Night
1499       1500         156         129 2025-05-10 23:01:51       Night

[1500 rows x 5 columns]


* Grouping and Aggregation: Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.

In [70]:
grouped = merge_df.groupby("Genre")
print(grouped)
summary = merge_df.groupby("Genre").agg({
    # "Minutes_Played": "sum",
    "length_in_ms": "mean",
    "listen_id": "unique"
})
print(summary)

             length_in_ms                                          listen_id
Genre                                                                       
Classical   270042.229167  [31, 66, 131, 158, 200, 371, 498, 531, 534, 59...
Electronic  218211.597796  [1, 3, 15, 27, 35, 45, 55, 61, 69, 80, 109, 12...
Hip-Hop     229298.519298  [5, 11, 46, 50, 103, 126, 161, 189, 192, 196, ...
Jazz        204591.538922  [70, 78, 82, 97, 99, 127, 178, 205, 218, 223, ...
Pop         222772.086806  [6, 8, 25, 36, 44, 52, 64, 89, 92, 96, 119, 12...
Rock        226570.799427  [48, 60, 63, 73, 76, 87, 88, 111, 118, 129, 14...


* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.

In [68]:
pivot = pd.pivot_table(
    users,
    values="user_id",
    index='User Age Group',
    columns="subscription_tier",
    aggfunc="sum",
)
print(pivot)

subscription_tier  Free  Premium
User Age Group                  
18-24              3818     1208
25-34              2230     1027
35-44              3154      655
45-54              2814     1846
55+                2518      830
